# KTIR Latency Estimation Demo

This notebook shows how to use `ktir_cpu` to get cycle-approximate latency estimates
for KTIR kernels — no Spyre hardware required.

**What we cover:**
1. Load a kernel and run it (regex parser, no extra dependencies)
2. Inspect the `LatencyReport` — cycles, time, bottleneck, roofline
3. Compare two kernels (matmul vs softmax)
4. Multi-kernel roofline: matmul, softmax, sdpa, paged attention
5. Per-op trace: paged attention (indirect K/V loads via block_tables)
6. Tune `HardwareConfig` to model different hardware targets
7. Using the MLIR frontend parser (optional, requires `mlir_ktdp`)


## Setup

```bash
uv sync --group dev   # installs jupyter + matplotlib
```

Then launch:

```bash
uv run jupyter notebook notebooks/latency_demo.ipynb
```

In [ ]:
import os, pathlib
import numpy as np
from ktir_cpu.interpreter import KTIRInterpreter
from ktir_cpu.latency import HardwareConfig

# Resolve paths relative to the repo root regardless of launch directory.
# The notebook lives in notebooks/, so the repo root is one level up.
_REPO = pathlib.Path(__file__).parent.parent if "__file__" in dir() else pathlib.Path.cwd()
if not (_REPO / "examples" / "latency").exists():
    _REPO = _REPO.parent   # launched from notebooks/
_LATENCY = _REPO / "examples" / "latency"

In [ ]:
import matplotlib.pyplot as plt

def plot_roofline(hw: HardwareConfig, kernels: list[tuple[str, object]],
                  granularity: str, title: str | None = None) -> None:
    """Roofline for ONE granularity, on its own figure (chip and per-core are
    never mixed). Pulls report.chip_roofline() or report.core_roofline().
    y is normalised (throughput as fraction of peak, in [0,1]); roof =
    min(1, AI/ridge) with ridge read from the report; one subplot per dominant unit."""
    assert granularity in ("chip", "core")
    rfs = [(name, rep.chip_roofline() if granularity == "chip" else rep.core_roofline())
           for name, rep in kernels]
    dom_key = "dominant_unit" if granularity == "chip" else "core_dominant_unit"
    ai_key  = "AI" if granularity == "chip" else "core_AI"

    from collections import defaultdict
    groups: dict[str, list] = defaultdict(list)
    for name, rf in rfs:
        groups[rf[dom_key]].append((name, rf))

    n_panels = len(groups)
    fig, axes = plt.subplots(n_panels, 1, figsize=(7, 4 * n_panels), squeeze=False)
    fig.suptitle(title or f"Roofline — {granularity}-level", fontsize=12)
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for row, (unit, kgroup) in enumerate(groups.items()):
        ax = axes[row][0]
        ridge = kgroup[0][1]["ridge"]          # granularity-invariant; from the report
        xs = [rf[ai_key] for _, rf in kgroup]
        finite = [x for x in xs if x != float("inf")]
        ai_max = max([ridge] + finite) * 1.3

        ai_range = np.linspace(0, ai_max, 400)
        roof = np.minimum(ai_range / ridge, 1.0)
        ax.plot(ai_range, roof, color="gray", linewidth=2, label="Roofline (norm)")
        ax.axvline(ridge, color="gray", linestyle="--", linewidth=1, alpha=0.6)
        ax.text(ridge * 1.02, 0.5, f"ridge={ridge:.1f} F/B", color="gray", fontsize=8, va="center")

        for i, (name, rf) in enumerate(kgroup):
            c = colors[i % len(colors)]
            if granularity == "chip":
                x, y = rf["AI"], rf["compute_throughput"]
            else:
                u = rf["units"][unit]
                x = rf["core_AI"]
                y = u["achieved_gflops"] / u["peak_gflops"] if u["peak_gflops"] else 0.0
            if x != float("inf"):
                ax.scatter(x, y, s=100, zorder=5, color=c, edgecolor="black", linewidth=0.5,
                           label=f"{name} (AI={x:.2f}, {y:.0%})")

        ax.set_xlim(left=0, right=ai_max)
        ax.set_ylim(bottom=0, top=1.1)
        ax.set_xlabel("Arithmetic intensity (FLOP/B)", fontsize=10)
        ax.set_ylabel("Throughput (fraction of peak)", fontsize=10)
        ax.set_title(f"Dominant unit: {unit}", fontsize=10)
        ax.legend(fontsize=8, loc="upper left")
        ax.grid(True, linestyle=":", alpha=0.4)

    plt.tight_layout()
    plt.show()


## 1. Matmul — load, run, report

**Kernel:** `examples/latency/matmul_small.mlir`  
Grid `[2, 2]` (4 cores), each core accumulates an 8×32 output tile over K=64 steps.

In [ ]:
hw = HardwareConfig()

interp = KTIRInterpreter(latency_config=hw)
interp.load(str(_LATENCY / "matmul_small.mlir"))

In [ ]:
M, K, N = 16, 64, 64
a = np.random.rand(M, K).astype(np.float16)
b = np.random.rand(K, N).astype(np.float16)
c = np.zeros((M, N), dtype=np.float16)

outputs = interp.execute_function(
    "matmul_kernel_small",
    a_ptr=a, b_ptr=b, c_ptr=c,
    K=K, BLOCK_SIZE_M=8, BLOCK_SIZE_N=32, BLOCK_SIZE_K=32,
)

# Verify correctness against NumPy
expected = a @ b
print("Max abs error:", np.max(np.abs(outputs["c_ptr"].astype(np.float32) - expected.astype(np.float32))))

In [ ]:
report = interp.get_latency_report()
print(report)

### Reading the report

Two methods, one per granularity: **`report.chip_roofline()`** (whole chip as one unit; plain NCU-style names)
and **`report.core_roofline()`** (critical-path core; `core_*` names). `report.roofline()` returns both merged.

**General**

| Field | Meaning |
|---|---|
| `kernel_cycles` | Max total cycles across all cores (critical path) |
| `kernel_time_us` | Cycles ÷ clock (default 1 GHz → 1 cycle = 1 ns) |
| `bottleneck` | Which category dominates the critical-path core: `compute`, `memory`, or `comm` |
| Per-core table | `compute_cycles`, `memory_cycles`, `comm_cycles` per core |

**Chip-level** (whole chip as one unit; aggregates over all cores, no `÷ num_cores` per-core inputs)

| Field | Meaning |
|---|---|
| `AI` | `Σ FLOP / Σ HBM bytes` over all cores (FLOP/B) — the roofline x-axis |
| `compute_throughput` | `Σ FLOP / elapsed / chip_peak` — compute SOL (fraction of chip compute peak); Nsight *Compute (SM) Throughput %* analogue |
| `dram_throughput` | `Σ HBM bytes / (hbm_bw × elapsed)` — memory SOL (fraction of aggregate HBM bandwidth); `hbm_bw` read directly from `hbm_bandwidth_tb_s` |
| `mean_core_active_frac` | `Σ core.total_cycles / (num_cores × elapsed_cycles)` — time-based core occupancy (active = compute + memory + comm); Nsight *SM Active %* analogue |
| `grid_coverage` | `cores_active / num_cores` — **spatial** dispatch coverage (not core busyness; *not* Nsight *SM Active %* — that role is `mean_core_active_frac`) |

**Per-core** (critical-path core; `core_` prefix)

| Field | Meaning |
|---|---|
| `core_AI` | dominant unit FLOPs ÷ HBM bytes on the critical core (FLOP/B) |
| `core_efficiency` | `achieved / ceiling` for the dominant unit — per-active-core throughput vs the roofline ceiling at this kernel\'s AI |
| `core_dominant_unit` | `systolic` or `simd` — the unit with the most compute cycles |
| `ridge_point` (per-unit) | AI where BW ceiling meets compute ceiling — left = memory-bound, right = compute-bound |
| Per-unit `peak_gflops` / `chip_peak_gflops` | per-core flat peak / `× num_cores` chip flat peak |
| Per-unit `compute_throughput` | this unit\'s chip-wide throughput (the per-unit detail behind the chip-level `compute_throughput`) |


In [ ]:
print(f"Kernel cycles : {report.kernel_cycles:,.0f}")
print(f"Kernel time   : {report.kernel_time_us:.3f} us")
print(f"Bottleneck    : {report.bottleneck}")

core = report.core_roofline()
dom = core["core_dominant_unit"]
u = core["units"][dom]
print(f"\nPer-core roofline  (critical-path core; dominant unit: {dom})")
print(f"  core_AI         : {core['core_AI']:.2f} FLOP/B")
print(f"  ridge_point     : {u['ridge_point']:.2f} FLOP/B")
print(f"  core_efficiency : {core['core_efficiency']:.1%}   (per-active-core, achieved/ceiling)")
print(f"  Kernel is {'compute-bound' if core['core_AI'] >= u['ridge_point'] else 'memory-bound'} (per-core view)")


## 2. Per-core breakdown

In [ ]:
per_core = report.per_core_summary()
header = f"{'core':>4}  {'compute':>12}  {'memory':>12}  {'comm':>12}  {'total':>12}"
print(header)
print("-" * len(header))
for row in per_core:
    print(f"{row['core_id']:>4}  {row['compute_cycles']:>12.0f}  "
          f"{row['memory_cycles']:>12.0f}  {row['comm_cycles']:>12.0f}  "
          f"{row['total_cycles']:>12.0f}")

## 3. Softmax — compare with matmul

In [ ]:
softmax_interp = KTIRInterpreter(latency_config=hw)
softmax_interp.load(str(_LATENCY / "softmax_small.mlir"))

n_rows = 64
input_data  = np.random.randn(n_rows, 64).astype(np.float16)
output_data = np.zeros((n_rows, 64), dtype=np.float16)

softmax_interp.execute_function(
    "softmax_kernel_small",
    output_ptr=output_data, input_ptr=input_data,
    n_rows=n_rows,
)

softmax_report = softmax_interp.get_latency_report()
print(softmax_report)

In [ ]:
# Per-core comparison (critical-path core)
for name, r in [("matmul", report), ("softmax", softmax_report)]:
    core = r.core_roofline()
    print(f"{name:10s} cycles={r.kernel_cycles:6.0f}  bottleneck={r.bottleneck:8s}  "
          f"core_AI={core['core_AI']:.2f} F/B  core_eff={core['core_efficiency']:.1%}  "
          f"dom={core['core_dominant_unit']}")


## 4. Multi-kernel roofline

Run four kernels through the latency estimator and plot them together:
- **matmul_small** and **sdpa** — systolic-dominant (matmul ops)
- **softmax** and **indexed_add** — simd-dominant (elementwise / transcendental ops)

In [ ]:
_TRITON = _REPO / "examples" / "triton-ktir"

def _run(path, func_name, tensor_kwargs, scalar_kwargs={}):
    interp = KTIRInterpreter(latency_config=hw)
    interp.load(str(path))
    interp.execute_function(func_name, **tensor_kwargs, **scalar_kwargs)
    return interp.get_latency_report()

rng = np.random.default_rng(0)

sdpa_report = _run(
    _TRITON / "sdpa_2d.mlir", "sdpa_kernel_2d",
    dict(q_ptr      = rng.standard_normal((32, 64)).astype(np.float16),
         k_ptr      = rng.standard_normal((32, 64)).astype(np.float16),
         v_ptr      = rng.standard_normal((32, 64)).astype(np.float16),
         output_ptr = np.zeros((32, 64), dtype=np.float16)),
)

paged_attn_report = _run(
    _TRITON / "paged_attention.mlir", "kernel_unified_attention_spyre_2d",
    dict(output_ptr       = np.zeros((8, 32, 128), dtype=np.float16),
         query_ptr        = rng.standard_normal((8, 32, 128)).astype(np.float16),
         key_cache_ptr    = rng.standard_normal((64, 16, 8, 128)).astype(np.float16),
         value_cache_ptr  = rng.standard_normal((64, 16, 8, 128)).astype(np.float16),
         block_tables_ptr = rng.integers(0, 64, size=(1, 16), dtype=np.int32)),
    dict(cur_batch_start_index=0, block_table_offset=0,
         num_tiles=8, context_len=120, scale=0.08838834764831843),
)

kernels = [("matmul_small", report), ("softmax", softmax_report),
           ("sdpa", sdpa_report), ("paged_attn", paged_attn_report)]

# Per-core comparison (critical-path core)
for name, r in kernels:
    core = r.core_roofline()
    print(f"{name:15s} cycles={r.kernel_cycles:6.0f}  bottleneck={r.bottleneck:8s}  "
          f"core_AI={core['core_AI']:6.2f} F/B  dom={core['core_dominant_unit']:8s}  "
          f"core_eff={core['core_efficiency']:.1%}")


### Per-op trace: paged attention

Paged attention uses `construct_indirect_access_tile` to load K and V tiles via
`block_tables` — an indirection table mapping sequence positions to KV-cache pages.
Re-running with `trace_latency=True` and grouping by op type shows where cycles go.

**Scenario:** 8 query tokens attending over 120 context tokens (`context_len=120,
num_tiles=8`). This resembles **chunked prefill or speculative decoding** — not a
pure prefill (where `num_tokens == context_len`) and not single-token decode.


In [ ]:
from collections import defaultdict

trace_interp = KTIRInterpreter(latency_config=hw, trace_latency=True)
trace_interp.load(str(_TRITON / "paged_attention.mlir"))
trace_interp.execute_function(
    "kernel_unified_attention_spyre_2d",
    output_ptr       = np.zeros((8, 32, 128), dtype=np.float16),
    query_ptr        = rng.standard_normal((8, 32, 128)).astype(np.float16),
    key_cache_ptr    = rng.standard_normal((64, 16, 8, 128)).astype(np.float16),
    value_cache_ptr  = rng.standard_normal((64, 16, 8, 128)).astype(np.float16),
    block_tables_ptr = rng.integers(0, 64, size=(1, 16), dtype=np.int32),
    cur_batch_start_index=0, block_table_offset=0,
    num_tiles=8, context_len=120, scale=0.08838834764831843,
)
trace_report = trace_interp.get_latency_report()
core0 = trace_report.counters[0]

# Group by (op_type, category), sum cycles and count occurrences
groups = defaultdict(lambda: [0.0, 0])
for e in core0.trace:
    groups[(e.op_type, e.category)][0] += e.cycles
    groups[(e.op_type, e.category)][1] += 1

print(f"{'Op':<45}  {'Category':<25}  {'Count':>5}  {'Cycles':>10}")
print("-" * 95)
for (op, cat), (cyc, cnt) in sorted(groups.items(), key=lambda x: -x[1][0]):
    if cyc > 0 or cnt > 1:
        print(f"{op:<45}  {cat:<25}  {cnt:>5}  {cyc:>10.1f}")
print("-" * 95)
print(f"{'TOTAL':<77}  {core0.total_cycles:>10.1f}")
print(f"  compute={core0.compute_cycles:.1f}  memory={core0.memory_cycles:.1f}  comm={core0.comm_cycles:.1f}")


In [ ]:
_kernels = [("matmul_small", report), ("sdpa", sdpa_report),
            ("softmax", softmax_report), ("paged_attn", paged_attn_report)]

plot_roofline(hw, _kernels, granularity="core", title="Per-core roofline — four kernels")


---

## Chip-level analysis — the whole chip as one unit

Everything above is the **per-core** analysis (the critical core vs its own ceiling). What follows is a
**separate analysis at a different granularity**: it collapses the chip to a single unit (the way Nsight
views a device) and asks *how much of the **entire chip** did this kernel use, and what bounds it* — a
question per-core cannot answer.

**Why chip-level is meaningful (and per-core is not enough).** A core can be **~99% efficient**
(`core_efficiency`) while the **whole chip sits mostly idle**. The chip metrics make that visible:

- `grid_coverage` / `mean_core_active_frac` — how many cores actually did work / for how much of the time.
- `dram_throughput` — whether the kernel saturates the **whole chip's** HBM bandwidth (device memory-bound).
- `compute_throughput` — the whole chip's compute utilisation vs its flat peak.

Convention: plain NCU-style names (`AI`, `compute_throughput`, …) are **chip-level**; a `core_` prefix is
**per-core**. The two are never compared on the same axes — their denominators differ.

In [ ]:
# Chip-level comparison (whole chip as one unit)
kernels = [("matmul_small", report), ("softmax", softmax_report),
           ("sdpa", sdpa_report), ("paged_attn", paged_attn_report)]

print("== Chip-level (whole chip as one unit) ==")
for name, r in kernels:
    chip = r.chip_roofline()
    print(f"{name:15s} AI={chip['AI']:6.2f} F/B  compute_thru={chip['compute_throughput']:6.1%}  "
          f"dram_thru={chip['dram_throughput']:6.1%}  mean_core_active={chip['mean_core_active_frac']:5.1%}  "
          f"grid_cov={chip['grid_coverage']:5.1%}  ({chip['cores_active']}/{chip['num_cores']} cores)")


In [ ]:
# Chip-level roofline — SEPARATE figure from the per-core one above
plot_roofline(hw, kernels, granularity="chip", title="Chip-level roofline — four kernels")


**Reading the chip metrics — what per-core hid**

- Kernels that fill only a few cores show low `grid_coverage` / `mean_core_active_frac` and tiny
  `compute_throughput` / `dram_throughput`, *even when* `core_efficiency` is ~99%: the busy core is
  efficient, but **most of the chip is idle**. Per-core alone would read as "great".
- Kernels that fill all cores and push `dram_throughput` toward 100% are **device HBM-bound**: they
  saturate the whole chip's memory bandwidth — a chip-level statement the per-core view cannot make.

## 5. Tuning HardwareConfig

`HardwareConfig` lets you model different hardware targets by adjusting bandwidth, clock, and SIMD width.
All fields default to reasonable approximations — override with real Spyre specs when available.

In [ ]:
# High-bandwidth variant: 4× HBM bandwidth
hw_hbw = HardwareConfig(
    num_cores=32,
    clock_ghz=1.5,
    hbm_bandwidth_tb_s=4.0,   # 4 TB/s aggregate
    simd_elements_per_cycle=128,
)

interp_hbw = KTIRInterpreter(latency_config=hw_hbw)
interp_hbw.load(str(_LATENCY / "softmax_small.mlir"))
interp_hbw.execute_function(
    "softmax_kernel_small",
    output_ptr=np.zeros((64, 64), dtype=np.float16),
    input_ptr=np.random.randn(64, 64).astype(np.float16),
    n_rows=64,
)
r_hbw = interp_hbw.get_latency_report()

print(f"Default HW  : {softmax_report.kernel_cycles:.0f} cycles  bottleneck={softmax_report.bottleneck}")
print(f"High-BW HW  : {r_hbw.kernel_cycles:.0f} cycles  bottleneck={r_hbw.bottleneck}")

## 6. MLIR frontend parser (optional)

By default `KTIRInterpreter` uses the built-in regex parser.
The MLIR frontend uses the real MLIR C++ bindings for full-fidelity parsing.

See the [README](../README.md#mlir-frontend-bindings-optional) for build and install instructions.

In [ ]:
try:
    from ktir_cpu.mlir_frontend.parser import MLIRFrontendParser
    MLIRFrontendParser()  # raises ImportError if mlir_ktdp not installed
    _mlir_available = True
except ImportError:
    _mlir_available = False
    print("mlir_ktdp not installed — skipping MLIR frontend cells.")
    print("See the install instructions above.")

In [ ]:
if _mlir_available:
    # Pass MLIRFrontendParser directly to the interpreter
    mlir_interp = KTIRInterpreter(
        latency_config=hw,
        parser=MLIRFrontendParser(),
    )
    mlir_interp.load(str(_LATENCY / "matmul_small.mlir"))

    M, K, N = 16, 64, 64
    mlir_interp.execute_function(
        "matmul_kernel_small",
        a_ptr=np.random.rand(M, K).astype(np.float16),
        b_ptr=np.random.rand(K, N).astype(np.float16),
        c_ptr=np.zeros((M, N), dtype=np.float16),
        K=K, BLOCK_SIZE_M=8, BLOCK_SIZE_N=32, BLOCK_SIZE_K=32,
    )

    mlir_report = mlir_interp.get_latency_report()
    print(mlir_report)

In [ ]:
if _mlir_available:
    print(f"regex  cycles={report.kernel_cycles:.0f}  AI={report.core_roofline()['core_AI']:.2f}")
    print(f"mlir   cycles={mlir_report.kernel_cycles:.0f}  AI={mlir_report.core_roofline()['core_AI']:.2f}")
    print("Results are identical — the two parsers produce the same IR.")


## Summary

```python
from ktir_cpu.interpreter import KTIRInterpreter
from ktir_cpu.latency import HardwareConfig

hw = HardwareConfig()                        # or HardwareConfig(clock_ghz=1.5, ...)
interp = KTIRInterpreter(latency_config=hw)  # add parser=MLIRFrontendParser() for MLIR
interp.load("path/to/kernel.mlir")           # or inline MLIR text

interp.execute_function("kernel_name", arg=tensor, ...)

report = interp.get_latency_report()
print(report)                        # human-readable table + roofline
report.kernel_cycles                 # float
report.kernel_time_us                # float
report.bottleneck                    # 'compute' | 'memory' | 'comm'
report.per_core_summary()            # list[dict] — per-core breakdown
report.chip_roofline()               # dict — AI, compute/dram_throughput, mean_core_active_frac, grid_coverage
report.core_roofline()               # dict — core_AI, core_efficiency, units, ...
report.roofline()                    # dict — chip + core merged
```